# RAG baseline evaluation (Ocean Research Assistant)

This notebook runs your 52-question `eval_set.json` against a RAG pipeline and gives you **numbers**, so every later change (chunk size, `top_k`, reranking, ...) can be judged by data instead of by feel.

**Run the cells top to bottom.** It has three parts:

| Part | What it measures | Needs an API key? |
|---|---|---|
| **A. Retrieval** | Did the right PDF page reach the top-k chunks? (`sources` field) | No |
| **B. Generation** | Did the answer contain the right facts? (`key_facts` field) | No with local Ollama (default); yes with Gemini |
| **C. Experiments** | Change one setting, re-run, compare to baseline | No (retrieval only) |

**Do Part A first.** If retrieval is bad, generation can't be good, and Part A is free and takes a minute.

### Folder layout expected
```
ocean-rag/
  eval_baseline.ipynb      <- this notebook
  .env                     <- GEMINI_API_KEY=...   (only if LLM_BACKEND = "gemini")
  data/
    RAG_doc1.pdf  RAG_doc2.pdf  RAG_doc3.pdf  RAG_doc4.pdf
    eval_set.json
```

In [ ]:
# Run once, then you can comment this out. (Use the same venv/kernel as your Phase 5 notebook.)
%pip install -q pypdf sentence-transformers numpy pandas google-genai python-dotenv requests

## 0. Settings
Everything you will want to tweak lives in this one cell.

In [ ]:
from pathlib import Path
import os, re, json, time, unicodedata
import numpy as np
import pandas as pd

DATA_DIR    = Path("data")                 # folder with the 4 PDFs + eval_set.json
EVAL_PATH   = DATA_DIR / "eval_set.json"
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

# which file is which document id used inside eval_set.json
DOC_FILES = {
    "athul2025":      "RAG_doc1.pdf",   # LSTM sea level anomaly forecasting
    "shenoi2020":     "RAG_doc2.pdf",   # Are we ready for a major tsunami?
    "manneela2016":   "RAG_doc3.pdf",   # Recent advances in ITEWS
    "copernicus2021": "RAG_doc4.pdf",   # Copernicus sea level product user guide
}
# cover pages with no real content (IOP "you may also like" page, JSTOR cover page)
SKIP_PAGES = {("athul2025", 1), ("shenoi2020", 1)}

# --- pipeline settings (the knobs you will experiment with) ---
CHUNK_SIZE = 800      # characters per chunk
OVERLAP    = 150      # characters shared between neighbouring chunks
TOP_K      = 5        # chunks handed to the LLM
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

# --- LLM for Part B ---
# "ollama": free, unlimited, runs locally (needs the Ollama app running + `ollama pull qwen3.5:9b`). ~50 s/question on CPU.
# "gemini": fast, but the free tier allows only 20 requests/day per model (a full run needs 52).
LLM_BACKEND  = "ollama"
OLLAMA_MODEL = "qwen3.5:9b"                                   # beat llama3 and gemma4:12b on this eval set, and was fastest
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")   # gemini-2.5-flash was retired (404); override via .env
LLM_MODEL    = OLLAMA_MODEL if LLM_BACKEND == "ollama" else GEMINI_MODEL
# Keep the same LLM for every experiment: scores from different LLMs are not comparable.

## 1. Load the PDFs, page by page

We keep the **page number** with every piece of text. That is what lets us check retrieval against the `sources` field in the eval set (`pdf_page` is the page number as the PDF viewer counts it, starting at 1).

Cleaning we do on purpose:
- remove the JSTOR footer ("This content downloaded from ... about.jstor.org/terms") that is printed on every page of `RAG_doc2.pdf`; otherwise it becomes noise that matches almost any query
- skip the two cover pages
- rejoin words hyphenated across lines (`informa-\ntion` -> `information`)

In [ ]:
from pypdf import PdfReader

JSTOR = re.compile(r"This content downloaded from.*?about\.jstor\.org/terms", re.S)

def clean(text):
    text = JSTOR.sub(" ", text)
    text = re.sub(r"-\n(?=[a-z])", "", text)   # un-hyphenate line breaks
    text = re.sub(r"\s+", " ", text)           # collapse whitespace
    return text.strip()

pages = []
for doc, filename in DOC_FILES.items():
    reader = PdfReader(DATA_DIR / filename)
    for page_no, page in enumerate(reader.pages, start=1):   # page_no starts at 1, like pdf_page
        if (doc, page_no) in SKIP_PAGES:
            continue
        text = clean(page.extract_text() or "")
        if len(text) > 50:
            pages.append({"doc": doc, "page": page_no, "text": text})

pd.DataFrame(pages).groupby("doc").agg(pages=("page", "count"), characters=("text", lambda s: s.str.len().sum()))

## 2. Chunk, embed, retrieve

This is the same idea as your Phase 5 pipeline: **document -> chunks -> embeddings -> cosine similarity -> top-k chunks**.
The corpus is only a few hundred chunks, so plain NumPy is enough (no FAISS needed here).

Chunks are cut **inside a page**, so every chunk knows exactly which page it came from.

In [ ]:
def make_chunks(pages, size, overlap):
    chunks, step = [], size - overlap
    for p in pages:
        t = p["text"]
        for start in range(0, max(len(t) - overlap, 1), step):
            piece = t[start:start + size].strip()
            if len(piece) >= 60:                       # drop tiny leftovers
                chunks.append({"doc": p["doc"], "page": p["page"], "text": piece})
    return chunks

In [ ]:
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer(EMBED_MODEL)

def embed(texts):
    # normalized vectors -> dot product == cosine similarity
    return _model.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)

def build_index(chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = make_chunks(pages, chunk_size, overlap)
    vecs = embed([c["text"] for c in chunks])
    return {"chunks": chunks, "vecs": vecs, "chunk_size": chunk_size, "overlap": overlap}

def retrieve(index, question, k=TOP_K):
    q = embed([question])[0]
    scores = index["vecs"] @ q
    top = np.argsort(-scores)[:k]
    return [dict(index["chunks"][i], score=float(scores[i])) for i in top]

index = build_index()
print(f"{len(index['chunks'])} chunks  (size={CHUNK_SIZE}, overlap={OVERLAP})")

In [ ]:
# Sanity check: what does retrieval return for one question?
for r in retrieve(index, "What accuracy are the bottom pressure recorders on tsunami buoys?", k=3):
    print(f"{r['doc']} p.{r['page']}  score={r['score']:.3f}\n   {r['text'][:200]}...\n")

## 3. Load the eval set

In [ ]:
eval_set  = json.load(open(EVAL_PATH, encoding="utf-8"))
questions = eval_set["questions"]
print(len(questions), "questions")
pd.Series([q["type"] for q in questions]).value_counts()

## Part A. Retrieval score (free, no API key)

For each answerable question we look at the top-k chunks and ask: **is any chunk from a page listed in `sources`?**

- `hit@1` : the very first chunk is from a correct page
- `hit@3` / `hit@5` : a correct page appears in the top 3 / top 5

Hand example: q01 has `sources = athul2025 p.2`. If your top-5 chunks are from `athul2025 p.4, shenoi2020 p.2, athul2025 p.2, ...` then hit@1 = 0, hit@3 = 1, hit@5 = 1.

The 3 unanswerable questions have no `sources`, so they are left out of this part.

In [ ]:
def eval_retrieval(index, k=5, verbose=False):
    rows = []
    for q in questions:
        if not q["sources"]:
            continue                                   # unanswerable: nothing to retrieve
        wanted = {(s["doc"], s["pdf_page"]) for s in q["sources"]}
        got = retrieve(index, q["question"], k=k)
        got_pages = [(g["doc"], g["page"]) for g in got]
        row = {"id": q["id"], "type": q["type"], "difficulty": q["difficulty"]}
        for kk in (1, 3, 5):
            if kk <= k:
                row[f"hit@{kk}"] = int(any(p in wanted for p in got_pages[:kk]))
        row["wanted"] = sorted(wanted)
        row["top_pages"] = got_pages
        rows.append(row)
    df = pd.DataFrame(rows)
    summary = {c: round(float(df[c].mean()), 3) for c in df.columns if c.startswith("hit@")}
    return df, summary

ret_df, ret_summary = eval_retrieval(index, k=5)
print("OVERALL:", ret_summary)
ret_df.groupby("type")[[c for c in ret_df.columns if c.startswith("hit@")]].mean().round(2)

### Look at the misses
This is where you learn the most. For every question where the right page did **not** make the top 5, you see what was expected and what was retrieved instead.

In [ ]:
misses = ret_df[ret_df["hit@5"] == 0]
print(f"{len(misses)} misses out of {len(ret_df)}\n")
qmap = {q["id"]: q for q in questions}
for _, r in misses.iterrows():
    print(r["id"], "-", qmap[r["id"]]["question"])
    print("   expected:", r["wanted"])
    print("   got     :", r["top_pages"], "\n")

## Part B. Generation score

By default this uses a local model through **Ollama** (free, no key, no limits): keep the Ollama app running and make sure `ollama list` shows `qwen3.5:9b`. A full run takes ~45 min on a laptop CPU. To use Gemini instead, set `LLM_BACKEND = "gemini"` in the settings cell and put `GEMINI_API_KEY=your_key` in `.env`.

For each question we send the top-k chunks + the question to Gemini, then score the answer:

- **Answerable questions:** `fact_score` = fraction of `key_facts` found in the answer.
  Hand example (q11): key_facts = [0.0266, 1.8751, 0.4488, 0.3044]. An answer mentioning only the first two scores 2/4 = **0.5**.
- **Unanswerable questions:** score is **1** only if the model says `NOT_IN_DOCUMENTS` (it must not invent an answer).

Matching ignores case, spaces and hyphens, so `3 d` matches `3-day` and `3 days`. Number words count as digits, so `three days` matches `3 d` and `twenty one` matches `21`.

An answerable question is always scored on its facts: a partial answer that ends with `NOT_IN_DOCUMENTS` for the missing part still gets credit for the facts it did find.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

_ONES = "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split()
_TENS = "twenty thirty forty fifty sixty seventy eighty ninety".split()
_NUM_WORDS = {w: i for i, w in enumerate(_ONES)} | {w: 20 + 10 * i for i, w in enumerate(_TENS)}

def words_to_digits(s):
    # "twenty one" / "twenty-one" -> "21", "three" -> "3"; \b keeps words like "often" or "tone" intact
    s = re.sub(r"\b(" + "|".join(_TENS) + r")[\s-]+(" + "|".join(_ONES[1:10]) + r")\b",
               lambda m: str(_NUM_WORDS[m.group(1)] + _NUM_WORDS[m.group(2)]), s)
    return re.sub(r"\b(" + "|".join(_NUM_WORDS) + r")\b", lambda m: str(_NUM_WORDS[m.group(1)]), s)

def norm(s):
    s = words_to_digits(unicodedata.normalize("NFKC", s).lower())
    s = s.replace("\u2013", "-").replace("\u2014", "-").replace("\u00d7", "x")
    return re.sub(r"[\s\-]+", "", s)

def fact_score(answer, facts):
    if not facts:
        return None
    a = norm(answer)
    return sum(norm(f) in a for f in facts) / len(facts)

PROMPT = '''You answer questions using ONLY the context passages below. Each passage is tagged with its source.
If the context does not contain the answer, reply with exactly: NOT_IN_DOCUMENTS
Otherwise answer concisely and keep the exact numbers from the context.

CONTEXT:
{context}

QUESTION: {question}
ANSWER:'''

def build_prompt(question, chunks):
    context = "\n\n".join(f"[{c['doc']} p.{c['page']}] {c['text']}" for c in chunks)
    return PROMPT.format(context=context, question=question)

class QuotaExhausted(Exception):
    """Daily free-tier quota used up (20 requests/day/model): waiting a few seconds won't help."""

def _retry_seconds(err):
    # Gemini puts the wait time in the error text, e.g. "Please retry in 16h20m6.5s" or "retryDelay': '58806s'"
    m = re.search(r"retryDelay'?:\s*'(\d+)s", str(err))
    return int(m.group(1)) if m else None

if LLM_BACKEND == "ollama":
    import requests
    OLLAMA_URL = "http://localhost:11434/api/generate"
    def generate(prompt):
        body = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
                "think": False,                             # no hidden reasoning: on a CPU it would take minutes per answer
                "options": {"temperature": 0,               # deterministic -> same answer every run, comparable scores
                            "num_ctx": 8192}}               # room for the prompt; Ollama silently cuts prompts longer than this
        try:
            r = requests.post(OLLAMA_URL, json=body, timeout=1800)
        except requests.ConnectionError:
            raise RuntimeError("Can't reach Ollama at localhost:11434. Start the Ollama app and re-run.") from None
        if r.status_code == 404:
            raise RuntimeError(f"Model {OLLAMA_MODEL} isn't installed. Run `ollama pull {OLLAMA_MODEL}` in a terminal.")
        r.raise_for_status()
        return r.json()["response"]
    print("Using local Ollama model:", OLLAMA_MODEL)
elif os.getenv("GEMINI_API_KEY"):
    from google import genai
    _client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    def generate(prompt):
        for attempt in range(4):
            try:
                return _client.models.generate_content(model=GEMINI_MODEL, contents=prompt).text or ""
            except Exception as e:
                code = getattr(e, "code", None)
                if code and 400 <= code < 500 and code != 429:
                    raise                                 # bad model name / key: retrying won't help, fail loudly
                wait = _retry_seconds(e)
                if code == 429 and ("PerDay" in str(e) or (wait or 0) > 120):
                    raise QuotaExhausted(f"daily quota for {GEMINI_MODEL} used up; retry in ~{(wait or 0) // 3600}h") from e
                wait = wait or 5 * (attempt + 1)          # per-minute limit or server error -> wait and retry
                print(f"   retry in {wait}s after error:", str(e)[:80])
                time.sleep(wait)
        raise RuntimeError("Gemini call failed 4 times in a row")
else:
    generate = None
    print("LLM_BACKEND is 'gemini' but no GEMINI_API_KEY found: Part B will be skipped. Parts A and C still work.")

In [ ]:
# Every answer is cached on disk, keyed by the model and settings that produced it, so an interrupted run
# continues where it stopped when you re-run this cell (with Gemini's free tier, 20 requests/day, a full
# run takes 3 days this way). Cached answers cost nothing.
CACHE_PATH = RESULTS_DIR / "gen_cache.json"
gen_cache = json.load(open(CACHE_PATH, encoding="utf-8")) if CACHE_PATH.exists() else {}

def eval_generation(index, k=TOP_K, sleep=None, only_ids=None):
    # sleep: seconds between calls, to stay under Gemini's per-minute limit (not needed for local Ollama)
    if sleep is None:
        sleep = 0 if LLM_BACKEND == "ollama" else 4.0
    rows = []
    for q in questions:
        if only_ids and q["id"] not in only_ids:
            continue
        key = f'{LLM_MODEL}|{index["chunk_size"]}|{index["overlap"]}|{k}|{q["id"]}'
        if key in gen_cache:
            answer = gen_cache[key]
        else:
            chunks = retrieve(index, q["question"], k=k)
            try:
                answer = generate(build_prompt(q["question"], chunks))
            except QuotaExhausted as e:
                print(f"Stopped early: {e}. Re-run this cell then to continue.")
                break
            gen_cache[key] = answer
            json.dump(gen_cache, open(CACHE_PATH, "w", encoding="utf-8"), indent=1)
            time.sleep(sleep)
        abstained = "NOT_IN_DOCUMENTS" in answer.upper().replace(" ", "_")
        if q["type"] == "unanswerable":
            score = 1.0 if abstained else 0.0
        else:
            score = fact_score(answer, q["key_facts"])   # a full abstention finds no facts -> 0 anyway
        rows.append({"id": q["id"], "type": q["type"], "difficulty": q["difficulty"],
                     "score": score, "abstained": abstained, "answer": answer.strip()})
    return pd.DataFrame(rows)

if generate:
    gen_df = eval_generation(index)                     # 52 questions: ~45 min with Ollama on CPU, a few min with Gemini
    n_total = len(questions)
    print(f"Answered {len(gen_df)}/{n_total} questions.  Average score:",
          round(gen_df["score"].mean(), 3) if len(gen_df) else None)
    if len(gen_df) < n_total:
        print("Partial run: don't save it as the baseline until all questions are answered.")
    if len(gen_df):
        display(gen_df.groupby("type")["score"].agg(["mean", "count"]).round(2))

### Save this as your baseline
Run this once. It stores the settings and the scores in `results/`, so later runs have something to be compared against.

In [ ]:
def save_run(name, index, ret_summary, gen_df=None, ret_df_=None):
    record = {
        "name": name,
        "settings": {"chunk_size": index["chunk_size"], "overlap": index["overlap"], "top_k": TOP_K,
                     "embed_model": EMBED_MODEL, "llm": LLM_MODEL},
        "retrieval": ret_summary,
        "generation_avg": None if gen_df is None else round(gen_df["score"].mean(), 3),
        "per_question": [] if gen_df is None else gen_df.drop(columns=["answer"]).to_dict("records"),
    }
    json.dump(record, open(RESULTS_DIR / f"{name}.json", "w"), indent=2)
    if gen_df is not None:
        gen_df.to_csv(RESULTS_DIR / f"{name}_answers.csv", index=False)       # read these to see WHY answers failed
    print("saved", RESULTS_DIR / f"{name}.json")

# only a complete Part B run counts; a partial one (quota hit) is saved as retrieval-only
full_gen = gen_df if "gen_df" in globals() and len(gen_df) == len(questions) else None
save_run("baseline", index, ret_summary, full_gen)

## Part C. Change one thing, compare

The rule: **change exactly one setting per experiment**, so you know what caused the difference.
Retrieval-only sweeps are free and fast, so start there. Here we vary chunk size:

In [ ]:
rows = []
for size, ov in [(400, 80), (800, 150), (1200, 200), (1600, 250)]:
    idx = build_index(size, ov)
    _, s = eval_retrieval(idx, k=5)
    rows.append({"chunk_size": size, "overlap": ov, "n_chunks": len(idx["chunks"]), **s})
sweep = pd.DataFrame(rows)
sweep

**How to read it:** pick the row with the best `hit@5` (and `hit@1` if you care about the first chunk being right).
Bigger chunks usually raise hit rate but add noise for the LLM, so confirm the winner with Part B before keeping it.

Next experiments, one at a time: `TOP_K` (3 vs 5 vs 8), overlap (0 vs 150 vs 300), a different embedding model.
When you change `CHUNK_SIZE`, `OVERLAP`, or `TOP_K` in the settings cell, re-run from section 2 down, then `save_run("chunk_1200", ...)` under a new name.